In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
from typing import Optional, Dict, Any, List, Tuple, Union
from PIL import Image
import matplotlib.pyplot as plt
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

In [ ]:
"""
Conditional UNet (Denoising Network)
从原仓库提取，保持 zero_module / 双条件注入 等工程细节
"""

def zero_module(module: nn.Module) -> nn.Module:
    """将所有可训练参数初始化为 0"""
    for p in module.parameters():
        p.detach().zero_()
    return module

def timestep_embedding(timesteps: torch.Tensor, dim: int, max_period: int = 10000) -> torch.Tensor:
    half = dim // 2
    freqs = torch.exp(
        -math.log(max_period) * torch.arange(0, half, dtype=torch.float32, device=timesteps.device) / half
    )
    args = timesteps[:, None].float() * freqs[None]
    embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
    return embedding

class Downsample(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.op = nn.Conv2d(channels, channels, 3, stride=2, padding=1)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.op(x)

class Upsample(nn.Module):
    def __init__(self, channels: int, use_conv: bool = True):
        super().__init__()
        self.use_conv = use_conv
        if self.use_conv:
            self.conv = nn.Conv2d(channels, channels, 3, stride=1, padding=1)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.interpolate(x, scale_factor=2, mode='nearest')
        if self.use_conv:
            x = self.conv(x)
        return x

class ConditionalResBlock(nn.Module):
    def __init__(
        self, in_ch: int, out_ch: int, time_emb_dim: int, cond_dim: int,
        dropout: float = 0.1, use_zero_module: bool = True
    ):
        super().__init__()
        self.norm1 = nn.GroupNorm(32, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_emb_proj = nn.Sequential(nn.SiLU(), nn.Linear(time_emb_dim, out_ch * 2))
        self.cond_emb_proj = nn.Sequential(nn.SiLU(), nn.Linear(cond_dim, out_ch * 2))
        self.norm2 = nn.GroupNorm(32, out_ch)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        if use_zero_module:
            self.conv2 = zero_module(self.conv2)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, time_emb, cond_emb):
        h = self.norm1(x)
        h = F.silu(h)
        h = self.conv1(h)
        time_out = self.time_emb_proj(time_emb)
        time_scale, time_shift = time_out.chunk(2, dim=1)
        time_scale = time_scale[:, :, None, None]
        time_shift = time_shift[:, :, None, None]
        cond_out = self.cond_emb_proj(cond_emb)
        cond_scale, cond_shift = cond_out.chunk(2, dim=1)
        cond_scale = cond_scale[:, :, None, None]
        cond_shift = cond_shift[:, :, None, None]
        h = self.norm2(h)
        h = h * (1 + time_scale) + time_shift
        h = h * (1 + cond_scale) + cond_shift
        h = F.silu(h)
        h = self.dropout(h)
        h = self.conv2(h)
        return self.skip(x) + h

class AttentionBlock(nn.Module):
    def __init__(self, channels: int, num_heads: int = 1):
        super().__init__()
        self.norm = nn.GroupNorm(32, channels)
        self.qkv = nn.Conv1d(channels, channels * 3, 1)
        self.attention = nn.MultiheadAttention(channels, num_heads, batch_first=True)
        self.proj = zero_module(nn.Conv1d(channels, channels, 1))
    def forward(self, x):
        b, c, h, w = x.shape
        x_reshape = x.reshape(b, c, -1)
        qkv = self.qkv(self.norm(x_reshape))
        q, k, v = qkv.chunk(3, dim=1)
        q, k, v = q.permute(0, 2, 1), k.permute(0, 2, 1), v.permute(0, 2, 1)
        attn_out, _ = self.attention(q, k, v)
        attn_out = attn_out.permute(0, 2, 1)
        attn_out = self.proj(attn_out)
        return x + attn_out.reshape(b, c, h, w)

class ConditionalUNet(nn.Module):
    def __init__(
        self, in_channels=3, model_channels=128, out_channels=3,
        num_res_blocks=2, attention_resolutions=(16, 8),
        channel_mult=(1, 2, 3, 4), cond_dim=512, dropout=0.1,
        use_zero_module=True, image_size=128
    ):
        super().__init__()
        self.image_size = image_size
        self.in_channels = in_channels
        self.model_channels = model_channels
        self.out_channels = out_channels
        self.channel_mult = channel_mult
        n_downsample = len(channel_mult) - 1
        min_divisible = 2 ** n_downsample
        if image_size % min_divisible != 0:
            raise ValueError(f"image_size ({image_size}) must be divisible by 2^{n_downsample} = {min_divisible}")
        time_emb_dim = model_channels * 4
        self.time_embed = nn.Sequential(
            nn.Linear(model_channels, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim),
        )
        self.input_conv = nn.Conv2d(in_channels, model_channels, 3, padding=1)
        self.down_blocks = nn.ModuleList()
        self.down_block_types = []
        skip_chs = [model_channels]
        ch = model_channels
        now_res = image_size
        for level, mult in enumerate(channel_mult):
            out_ch = model_channels * mult
            for _ in range(num_res_blocks):
                self.down_blocks.append(ConditionalResBlock(
                    ch, out_ch, time_emb_dim, cond_dim, dropout, use_zero_module
                ))
                self.down_block_types.append("res")
                ch = out_ch
                skip_chs.append(ch)
                if now_res in attention_resolutions:
                    self.down_blocks.append(AttentionBlock(ch))
                    self.down_block_types.append("attn")
            if level != len(channel_mult) - 1:
                self.down_blocks.append(Downsample(ch))
                self.down_block_types.append("down")
                skip_chs.append(ch)
                now_res //= 2
        self.mid_block1 = ConditionalResBlock(ch, ch, time_emb_dim, cond_dim, dropout, use_zero_module)
        self.mid_attn = AttentionBlock(ch)
        self.mid_block2 = ConditionalResBlock(ch, ch, time_emb_dim, cond_dim, dropout, use_zero_module)
        self.up_blocks = nn.ModuleList()
        self.up_block_types = []
        for level, mult in enumerate(reversed(channel_mult)):
            out_ch = model_channels * mult
            for _ in range(num_res_blocks + 1):
                self.up_blocks.append(ConditionalResBlock(
                    ch + skip_chs.pop(), out_ch, time_emb_dim, cond_dim, dropout, use_zero_module
                ))
                self.up_block_types.append("res")
                ch = out_ch
                if now_res in attention_resolutions:
                    self.up_blocks.append(AttentionBlock(ch))
                    self.up_block_types.append("attn")
            if level != len(channel_mult) - 1:
                self.up_blocks.append(Upsample(ch))
                self.up_block_types.append("up")
                now_res *= 2
        self.out = nn.Sequential(
            nn.GroupNorm(32, ch),
            nn.SiLU(),
            zero_module(nn.Conv2d(ch, out_channels, 3, padding=1)) if use_zero_module
            else nn.Conv2d(ch, out_channels, 3, padding=1),
        )

    def forward(self, x, t, cond):
        t_emb = timestep_embedding(t, self.model_channels)
        t_emb = self.time_embed(t_emb)
        h = self.input_conv(x)
        hs = [h]
        for block_type, block in zip(self.down_block_types, self.down_blocks):
            if block_type == "res":
                h = block(h, t_emb, cond)
                hs.append(h)
            elif block_type == "attn":
                h = block(h)
            elif block_type == "down":
                h = block(h)
                hs.append(h)
        h = self.mid_block1(h, t_emb, cond)
        h = self.mid_attn(h)
        h = self.mid_block2(h, t_emb, cond)
        for block_type, block in zip(self.up_block_types, self.up_blocks):
            if block_type == "res":
                h = block(torch.cat([h, hs.pop()], dim=1), t_emb, cond)
            elif block_type == "attn":
                h = block(h)
            elif block_type == "up":
                h = block(h)
        return self.out(h)


In [ ]:
"""
DDIM / DDPM Sampler
从原仓库提取，保持 spaced diffusion / clip_denoised 等细节
"""

def get_named_beta_schedule(schedule_name: str, num_diffusion_timesteps: int) -> np.ndarray:
    if schedule_name == "linear":
        scale = 1000 / num_diffusion_timesteps
        return np.linspace(scale * 0.0001, scale * 0.02, num_diffusion_timesteps, dtype=np.float64)
    elif schedule_name == "cosine":
        return betas_for_alpha_bar(
            num_diffusion_timesteps,
            lambda t: math.cos((t + 0.008) / 1.008 * math.pi / 2) ** 2,
        )
    else:
        raise ValueError(f"Unknown beta schedule: {schedule_name}")

def betas_for_alpha_bar(num_diffusion_timesteps: int, alpha_bar, max_beta: float = 0.999) -> np.ndarray:
    betas = []
    for i in range(num_diffusion_timesteps):
        t1 = i / num_diffusion_timesteps
        t2 = (i + 1) / num_diffusion_timesteps
        betas.append(min(1 - alpha_bar(t2) / alpha_bar(t1), max_beta))
    return np.array(betas, dtype=np.float64)

def space_timesteps(num_timesteps: int, section_counts: Union[str, List[int]]) -> List[int]:
    if isinstance(section_counts, str):
        if section_counts.startswith("ddim"):
            section_counts = [int(section_counts[4:])]
        else:
            raise ValueError(f"Unsupported spaced format: {section_counts}")
    if len(section_counts) == 1:
        num_sections = section_counts[0]
        stride = num_timesteps // num_sections
        used = set(range(0, num_timesteps, stride))
        if (num_timesteps - 1) not in used:
            used.add(num_timesteps - 1)
        return sorted(list(used))
    else:
        raise NotImplementedError("Only uniform spaced timesteps are supported now.")

class DDIMSampler:
    def __init__(
        self, num_timesteps=1000, beta_schedule="linear", use_timesteps=None,
        clip_denoised=True, pred_xstart_detach=True, device=None,
        custom_timestep_sampler=None
    ):
        self.num_timesteps = num_timesteps
        self.clip_denoised = clip_denoised
        self.pred_xstart_detach = pred_xstart_detach
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.use_custom_timestep_sampler = False
        self.custom_timestep_min = 0
        self.custom_timestep_max = num_timesteps
        self.custom_timestep_indices = None
        self.num_custom_timesteps = 0
        if custom_timestep_sampler is not None:
            assert len(custom_timestep_sampler) == 2
            self.custom_timestep_min = custom_timestep_sampler[0]
            self.custom_timestep_max = custom_timestep_sampler[1]
            self.use_custom_timestep_sampler = True
        betas = get_named_beta_schedule(beta_schedule, num_timesteps)
        self.betas = torch.from_numpy(betas).float().to(device)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = torch.cat([torch.tensor([1.0], device=device), self.alphas_cumprod[:-1]])
        self.alphas_cumprod_next = torch.cat([self.alphas_cumprod[1:], torch.tensor([0.0], device=device)])
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)
        self.sqrt_recip_alphas_cumprod = torch.sqrt(1.0 / self.alphas_cumprod)
        self.sqrt_recipm1_alphas_cumprod = torch.sqrt(1.0 / self.alphas_cumprod - 1)
        if use_timesteps is None:
            use_timesteps = list(range(num_timesteps))
        self.use_timesteps = sorted(list(use_timesteps))
        self.num_sample_steps = len(self.use_timesteps)
        self.timestep_map = {i: self.use_timesteps[i] for i in range(self.num_sample_steps)}
        if self.use_custom_timestep_sampler:
            self.custom_timestep_indices = [
                i for i, ts in enumerate(self.use_timesteps)
                if self.custom_timestep_min <= ts <= self.custom_timestep_max
            ]
            self.num_custom_timesteps = len(self.custom_timestep_indices)
            if self.num_custom_timesteps == 0:
                raise ValueError("No spaced timesteps fall in custom range")

    def _extract(self, arr, t, x_shape):
        out = arr[t].float()
        while len(out.shape) < len(x_shape):
            out = out.unsqueeze(-1)
        return out

    def q_sample(self, x_start, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_start)
        sqrt_alpha = self._extract(self.sqrt_alphas_cumprod, t, x_start.shape)
        sqrt_one_minus = self._extract(self.sqrt_one_minus_alphas_cumprod, t, x_start.shape)
        return sqrt_alpha * x_start + sqrt_one_minus * noise

    def _predict_xstart_from_eps(self, x_t, t, eps):
        return (
            self._extract(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t
            - self._extract(self.sqrt_recipm1_alphas_cumprod, t, x_t.shape) * eps
        )

    def p_mean_variance(self, model, x_t, t, model_kwargs=None):
        if model_kwargs is None:
            model_kwargs = {}
        model_output = model(x_t, t, **model_kwargs)
        pred_eps = model_output
        pred_xstart = self._predict_xstart_from_eps(x_t, t, pred_eps)
        if self.clip_denoised:
            pred_xstart = torch.clamp(pred_xstart, -1.0, 1.0)
        if self.pred_xstart_detach:
            pred_xstart = pred_xstart.detach()
        return pred_eps, pred_xstart

    def ddim_sample(self, model, x_t, t, model_kwargs=None, eta=0.0):
        pred_eps, pred_xstart = self.p_mean_variance(model, x_t, t, model_kwargs)
        alpha_bar = self._extract(self.alphas_cumprod, t, x_t.shape)
        alpha_bar_prev = self._extract(self.alphas_cumprod_prev, t, x_t.shape)
        sigma = eta * torch.sqrt((1 - alpha_bar_prev) / (1 - alpha_bar)) * torch.sqrt(1 - alpha_bar / alpha_bar_prev)
        noise = torch.randn_like(x_t)
        mean_pred = pred_xstart * torch.sqrt(alpha_bar_prev) + torch.sqrt(1 - alpha_bar_prev - sigma ** 2) * pred_eps
        nonzero_mask = ((t != 0).float()).view(-1, *([1] * (len(x_t.shape) - 1)))
        sample = mean_pred + nonzero_mask * sigma * noise
        return {"sample": sample, "pred_xstart": pred_xstart}

    def ddim_sample_loop(self, model, noise=None, shape=None, model_kwargs=None, device=None, progress=False):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        if noise is not None:
            x = noise
        else:
            assert shape is not None
            x = torch.randn(*shape, device=device)
        indices = list(range(self.num_sample_steps))[::-1]
        if progress:
            from tqdm import tqdm
            indices = tqdm(indices, desc="DDIM Sampling")
        for i in indices:
            t = torch.full((x.shape[0],), self.timestep_map[i], device=device, dtype=torch.long)
            with torch.no_grad():
                out = self.ddim_sample(model, x, t, model_kwargs=model_kwargs)
                x = out["sample"]
        return x

    def training_losses(self, model, x_start, model_kwargs=None):
        B = x_start.shape[0]
        device = x_start.device
        if self.use_custom_timestep_sampler:
            local_idx = torch.randint(0, self.num_custom_timesteps, (B,), device=device)
            t_idx = torch.tensor([self.custom_timestep_indices[i] for i in local_idx], device=device, dtype=torch.long)
        else:
            t_idx = torch.randint(0, self.num_sample_steps, (B,), device=device)
        t = torch.tensor([self.timestep_map[idx.item()] for idx in t_idx], device=device, dtype=torch.long)
        noise = torch.randn_like(x_start)
        x_t = self.q_sample(x_start, t, noise)
        model_output = model(x_t, t, **(model_kwargs or {}))
        loss = F.mse_loss(model_output, noise, reduction="none")
        loss = loss.mean(dim=list(range(1, len(loss.shape))))
        return {"loss": loss, "noise": noise, "x_t": x_t, "t": t}


In [ ]:
"""
FCN 语义编码器封装 + 图像预处理
"""

class ZSemExtractor(nn.Module):
    """
    从 FCN backbone 提取 z_sem，结构与原仓库一致。
    用户只需提供自己的 backbone（如 ResNet50）并加载权重。
    """
    def __init__(self, backbone, enc_output_dim=2048, cond_dim=512):
        super().__init__()
        self.backbone = backbone
        if enc_output_dim != cond_dim:
            self.diffae_enc_compress = nn.Sequential(
                nn.Linear(enc_output_dim, cond_dim),
                nn.SiLU(),
                nn.Linear(cond_dim, cond_dim),
            )
        else:
            self.diffae_enc_compress = nn.Identity()

    def forward(self, x):
        """
        Args:
            x: 经过 mmseg 预处理的图像 [B, 3, H, W]（ImageNet 归一化）
        Returns:
            z_sem: [B, cond_dim]
        """
        feats = self.backbone(x)
        feat = feats[-1] if isinstance(feats, (list, tuple)) else feats
        z = F.adaptive_avg_pool2d(feat, 1).flatten(1)
        z_sem = self.diffae_enc_compress(z)
        return z_sem

    def load_from_checkpoint(self, checkpoint_path, strict=False):
        """
        从训练好的 FCN 权重加载 backbone + compress。
        checkpoint 可以是 .pth 或 .ckpt，自动解析 state_dict。
        """
        ckpt = torch.load(checkpoint_path, map_location='cpu')
        state_dict = ckpt['state_dict'] if 'state_dict' in ckpt else ckpt
        self.load_state_dict(state_dict, strict=strict)
        print(f"Loaded checkpoint from {checkpoint_path}")


def preprocess_image(img_path, image_size=128, device='cuda'):
    """
    读取单张图片，执行与训练时一致的预处理：
      1. Resize -> image_size
      2. 转 Tensor
      3. ImageNet Normalize（mmseg 默认统计量）
      4. 转到 diffae 范围 [-1, 1]
    """
    from torchvision import transforms
    img = Image.open(img_path).convert('RGB')
    img = img.resize((image_size, image_size), Image.BILINEAR)
    to_tensor = transforms.ToTensor()
    x = to_tensor(img)  # [3, H, W], in [0,1]
    mean = torch.tensor([123.675, 116.28, 103.53]).view(3, 1, 1) / 255.0
    std = torch.tensor([58.395, 57.12, 57.375]).view(3, 1, 1) / 255.0
    x_norm = (x - mean) / std
    x_diffae = x_norm * 2.0 - 1.0
    return x_diffae.unsqueeze(0).to(device), x_norm.unsqueeze(0).to(device)


In [ ]:
# ================================================================
# 用户配置区 —— 请根据实际情况修改以下路径和参数
# ================================================================

# 1) FCN 权重路径（你训练好的模型 .pth / .ckpt）
FCN_CHECKPOINT_PATH = "./work_dirs/ablation_experiment/diffae_ade20k_customSampler/iter_80000.pth"

# 2) 待还原的单张图片路径
IMAGE_PATH = "./demo_images/sample_001.jpg"

# 3) 模型超参（必须与你训练 FCN 时使用的配置一致）
IMAGE_SIZE = 128
ENC_OUTPUT_DIM = 2048
COND_DIM = 512
UNET_CHANNELS = 128
UNET_CHANNEL_MULT = (1, 1, 2, 3)
NUM_RES_BLOCKS = 1
NUM_DIFFUSION_TIMESTEPS = 1000
BETA_SCHEDULE = "linear"
SPACED_TIMESTEPS = "ddim100"
CUSTOM_TIMESTEP_SAMPLER = [100, 500]   # 与原配置一致

# 4) 训练超参
TRAIN_STEPS = 2000
LR = 1e-4
LOG_INTERVAL = 100
SAVE_INTERVAL = 500
OUTPUT_DIR = "./single_image_ddpm_output"

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# ================================================================
# 初始化 Backbone 并加载 FCN 权重
# 
# 注意：这里以 torchvision.resnet50 为例。
# 如果你用的是 mmseg 的 ResNet，请替换为对应的 backbone 类，
# 并确保输出通道数为 ENC_OUTPUT_DIM（默认 2048）。
# ================================================================

from torchvision.models import resnet50

class ResNet50Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        base = resnet50(weights=None)
        self.conv1 = base.conv1
        self.bn1 = base.bn1
        self.relu = base.relu
        self.maxpool = base.maxpool
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x1 = self.layer1(x)
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        return [x1, x2, x3, x4]

backbone = ResNet50Backbone().to(device)
fcn_extractor = ZSemExtractor(backbone, enc_output_dim=ENC_OUTPUT_DIM, cond_dim=COND_DIM).to(device)

fcn_extractor.load_from_checkpoint(FCN_CHECKPOINT_PATH, strict=False)
fcn_extractor.eval()
for p in fcn_extractor.parameters():
    p.requires_grad = False

print("FCN 编码器已冻结，z_sem 将在训练时固定。")


In [ ]:
# ================================================================
# 初始化 UNet（zero_module=True，保证 zero 初始化）和 Sampler
# ================================================================

use_timesteps = space_timesteps(NUM_DIFFUSION_TIMESTEPS, SPACED_TIMESTEPS)

unet = ConditionalUNet(
    in_channels=3,
    model_channels=UNET_CHANNELS,
    out_channels=3,
    num_res_blocks=NUM_RES_BLOCKS,
    channel_mult=UNET_CHANNEL_MULT,
    cond_dim=COND_DIM,
    use_zero_module=True,
    image_size=IMAGE_SIZE,
).to(device)

sampler = DDIMSampler(
    num_timesteps=NUM_DIFFUSION_TIMESTEPS,
    beta_schedule=BETA_SCHEDULE,
    use_timesteps=use_timesteps,
    clip_denoised=True,
    pred_xstart_detach=True,
    device=device,
    custom_timestep_sampler=CUSTOM_TIMESTEP_SAMPLER,
)

optimizer = torch.optim.AdamW(unet.parameters(), lr=LR)

print(f"UNet 总参数量: {sum(p.numel() for p in unet.parameters()) / 1e6:.2f} M")
print("UNet 已 zero 初始化，准备单图训练...")


In [ ]:
# ================================================================
# 加载图片 -> 预处理 -> 提取固定 z_sem
# ================================================================

x_diffae, x_normalized = preprocess_image(IMAGE_PATH, image_size=IMAGE_SIZE, device=device)

with torch.no_grad():
    z_sem = fcn_extractor(x_normalized)

print(f"图片形状: {x_diffae.shape}")
print(f"z_sem 形状: {z_sem.shape}")
print(f"z_sem 范数: {z_sem.norm().item():.4f}")

def show_tensor(x, title=""):
    img = x[0].permute(1, 2, 0).cpu().numpy()
    img = (img + 1) / 2
    img = np.clip(img, 0, 1)
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.title(title)
    plt.axis('off')
    plt.show()

show_tensor(x_diffae, "原图 (DiffAE 范围 [-1,1])")


In [ ]:
# ================================================================
# 单图 DDIM 训练：固定 z_sem，只训练 UNet
# ================================================================

unet.train()
losses = []

for step in range(1, TRAIN_STEPS + 1):
    optimizer.zero_grad()

    diff_out = sampler.training_losses(
        unet, x_diffae, model_kwargs={"cond": z_sem}
    )
    loss = diff_out["loss"].mean()

    loss.backward()
    optimizer.step()

    losses.append(loss.item())

    if step % LOG_INTERVAL == 0:
        print(f"Step {step:04d} | Loss: {loss.item():.6f}")

    if step % SAVE_INTERVAL == 0 or step == 1:
        unet.eval()
        with torch.no_grad():
            x_rec = sampler.ddim_sample_loop(
                unet,
                noise=torch.randn_like(x_diffae),
                model_kwargs={"cond": z_sem},
                device=device,
            )
        unet.train()

        fig, axes = plt.subplots(1, 2, figsize=(8, 4))
        orig = (x_diffae[0].permute(1,2,0).cpu().numpy() + 1) / 2
        rec = (x_rec[0].permute(1,2,0).cpu().numpy() + 1) / 2
        rec = np.clip(rec, 0, 1)
        axes[0].imshow(np.clip(orig, 0, 1))
        axes[0].set_title("Original")
        axes[0].axis('off')
        axes[1].imshow(rec)
        axes[1].set_title(f"Reconstructed (step {step})")
        axes[1].axis('off')
        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/step_{step:05d}.png", dpi=150)
        plt.show()

print("\n训练完成！")


In [ ]:
# 绘制 loss 曲线
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel("Step")
plt.ylabel("MSE Loss")
plt.title("Single-Image DDIM Training Loss")
plt.grid(True)
plt.savefig(f"{OUTPUT_DIR}/loss_curve.png", dpi=150)
plt.show()


In [ ]:
# ================================================================
# 最终重建：从随机噪声出发，用训练好的 UNet 生成
# ================================================================

unet.eval()
with torch.no_grad():
    n_samples = 5
    recs = []
    for _ in range(n_samples):
        x_rec = sampler.ddim_sample_loop(
            unet,
            noise=torch.randn_like(x_diffae),
            model_kwargs={"cond": z_sem},
            device=device,
        )
        recs.append(x_rec)
    x_rec_mean = torch.stack(recs).mean(dim=0)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
orig = (x_diffae[0].permute(1,2,0).cpu().numpy() + 1) / 2
rec_single = (recs[0][0].permute(1,2,0).cpu().numpy() + 1) / 2
rec_avg = (x_rec_mean[0].permute(1,2,0).cpu().numpy() + 1) / 2

axes[0].imshow(np.clip(orig, 0, 1))
axes[0].set_title("Original")
axes[0].axis('off')
axes[1].imshow(np.clip(rec_single, 0, 1))
axes[1].set_title("Reconstructed (single sample)")
axes[1].axis('off')
axes[2].imshow(np.clip(rec_avg, 0, 1))
axes[2].set_title("Reconstructed (5-sample mean)")
axes[2].axis('off')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/final_comparison.png", dpi=150)
plt.show()


In [ ]:
# 保存训练好的 UNet 权重
torch.save(unet.state_dict(), f"{OUTPUT_DIR}/unet_single_image_trained.pth")
print(f"UNet 权重已保存至 {OUTPUT_DIR}/unet_single_image_trained.pth")
